# Evaluating LLMs on Summarization (Accuracy, ROUGE and Model Comparison)

Complete notebook covering Parts I–VIII. **Recommended:** `Runtime > Change runtime type > GPU (T4)`.

## Part I. Setup

In [ ]:
!pip install rouge_score==0.1.2 evaluate datasets nltk --quiet
!pip install -U accelerate --quiet

In [ ]:
import gc, os
import nltk
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import evaluate
from transformers import AutoTokenizer, T5ForConditionalGeneration, AutoModelForCausalLM

nltk.download("punkt")
nltk.download("punkt_tab")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
pd.set_option("display.max_colwidth", 120)

## Part II. Dataset Loading and Exploration
Upload `train.csv` and `test.csv` when prompted (or place them in `/content`).

In [ ]:
# Upload the files if they are not already in the working directory
if not (os.path.exists("train.csv") and os.path.exists("test.csv")):
    from google.colab import files
    files.upload()   # select train.csv and test.csv

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print("train:", train_df.shape, "| test:", test_df.shape)
print(train_df.columns.tolist())

In [ ]:
# Reproducible sampling to reduce the computational load
train_sample = train_df.sample(n=100, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=min(50, len(test_df)), random_state=42).reset_index(drop=True)

# First training example: article and reference summary
print("ARTICLE (prompt_text):\n", train_sample.loc[0, "prompt_text"][:1500], "...\n")
print("REFERENCE SUMMARY (prompt_title):\n", train_sample.loc[0, "prompt_title"])

In [ ]:
# Inspect the sampled DataFrames
print("=== TRAIN SAMPLE ==="); display(train_sample.head())
print("=== TEST SAMPLE ===");  display(test_sample.head())
print(train_sample.shape, test_sample.shape)

## Part III. Summarization with T5

In [ ]:
def batch_generator(items, batch_size):
    """Yield consecutive sublists of size `batch_size`."""
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def summarize_with_t5(articles, model_name="t5-small", batch_size=8,
                      max_input_length=512, max_new_tokens=64):
    """Generate summaries with a T5 model using batch processing."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device).eval()

    summaries = []
    for batch in batch_generator(list(articles), batch_size):
        # Tokenize the articles with the "summarize: " prefix
        inputs = tokenizer(
            ["summarize: " + a for a in batch],
            return_tensors="pt", padding=True,
            truncation=True, max_length=max_input_length,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                num_beams=4, early_stopping=True,
            )
        # Decode the generated token ids back to text
        summaries.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))

        # Free memory after every batch
        del inputs, output_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # Final cleanup
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return summaries

In [ ]:
articles = train_sample["prompt_text"].tolist()
references = train_sample["prompt_title"].tolist()

# Generate summaries for the training sample with t5-small
t5_small_summaries = summarize_with_t5(articles, model_name="t5-small")

# Show generated summaries next to the reference summaries
results_df = pd.DataFrame({
    "reference_summary": references,
    "t5_small_summary": t5_small_summaries,
})
results_df.head(10)

## Part IV. Accuracy Evaluation

In [ ]:
def accuracy_score_exact(predictions, references):
    """Exact-match accuracy (ignoring case and surrounding whitespace)."""
    matches = [p.strip().lower() == r.strip().lower() for p, r in zip(predictions, references)]
    return float(np.mean(matches))

acc = accuracy_score_exact(t5_small_summaries, references)
print(f"t5-small accuracy: {acc:.4f}")

**Interpretation.** Exact-match accuracy is practically **0** because a summary is free text: there are countless valid ways to summarize the same content, and accuracy only rewards character-by-character equality. It cannot tell a nearly perfect summary from a completely irrelevant one (both score 0), and it ignores synonyms, paraphrases and word order. That is why we need **partial-overlap** metrics such as ROUGE.

In [ ]:
# (Optional) Custom accuracy: fraction of reference words that appear in the prediction
def token_overlap_accuracy(predictions, references):
    scores = []
    for p, r in zip(predictions, references):
        ref_tokens = set(r.lower().split())
        pred_tokens = set(p.lower().split())
        scores.append(len(ref_tokens & pred_tokens) / max(len(ref_tokens), 1))
    return float(np.mean(scores))

print(f"Token-overlap accuracy: {token_overlap_accuracy(t5_small_summaries, references):.4f}")

## Part V. ROUGE Metric Implementation
**ROUGE** (*Recall-Oriented Understudy for Gisting Evaluation*) measures n-gram overlap (ROUGE-1, ROUGE-2), the longest common subsequence (ROUGE-L) and its summary-level version (ROUGE-Lsum) between a generated summary and a reference summary.

**Preprocessing:** `rougeLsum` expects **one sentence per line**, so we split each text into sentences with the NLTK sentence tokenizer (`sent_tokenize`) and join them with `"\n"`.

In [ ]:
# Load the ROUGE metric
rouge = evaluate.load("rouge")

def format_for_rouge(text):
    """One sentence per line, using the NLTK sentence tokenizer."""
    return "\n".join(nltk.sent_tokenize(text.strip()))

def compute_rouge_score(predictions, references, use_stemmer=True):
    # Put each sentence on its own line before scoring
    preds = [format_for_rouge(p) for p in predictions]
    refs = [format_for_rouge(r) for r in references]
    result = rouge.compute(predictions=preds, references=refs, use_stemmer=use_stemmer)
    return {k: round(float(v), 4) for k, v in result.items()}

print("t5-small ROUGE:", compute_rouge_score(t5_small_summaries, references))

## Part VI. Understanding ROUGE Scores

### 1. Exact match (predictions = references)

In [ ]:
print(compute_rouge_score(references, references))   # every score should be 1.0

### 2. Null predictions (empty summaries)

In [ ]:
print(compute_rouge_score([""] * len(references), references))   # every score should be 0.0

### 3. Effect of stemming

In [ ]:
pred = ["the cats are running quickly"]
ref  = ["the cat is run quick"]
print("Without stemming:", compute_rouge_score(pred, ref, use_stemmer=False))
print("With stemming   :", compute_rouge_score(pred, ref, use_stemmer=True))

With stemming, words such as *cats/cat* and *running/run* are reduced to the same stem and count as matches, so the score goes up.

### 4. N-gram analysis (ROUGE-1 vs ROUGE-2)

In [ ]:
ref = ["the quick brown fox jumps over the lazy dog"]
cases = {
    "Identical":                 "the quick brown fox jumps over the lazy dog",
    "Same words, shuffled":      "dog lazy the over jumps fox brown quick the",
    "Partial overlap (~50%)":    "the quick brown cat sleeps under a warm sun",
    "Only one consecutive pair": "quick brown",
    "No overlap":                "a completely different sentence appears here",
}
rows = []
for name, p in cases.items():
    s = compute_rouge_score([p], ref, use_stemmer=False)
    rows.append({"case": name, "rouge1": s["rouge1"], "rouge2": s["rouge2"], "rougeL": s["rougeL"]})
pd.DataFrame(rows)

ROUGE-1 only depends on the words; ROUGE-2 requires consecutive bigrams, so it drops sharply when the word order is changed even though the words are the same.

### 5. Symmetry with respect to predictions and references

In [ ]:
from rouge_score import rouge_scorer

pred = ["the cat sat on the mat today"]
ref  = ["the cat sat"]

print("pred->ref:", compute_rouge_score(pred, ref, use_stemmer=False))
print("ref->pred:", compute_rouge_score(ref, pred, use_stemmer=False))

# Precision and recall DO swap; the F1 score stays the same
scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=False)
a = scorer.score(ref[0], pred[0])["rouge1"]   # (target, prediction)
b = scorer.score(pred[0], ref[0])["rouge1"]
print(f"\nNormal  -> P={a.precision:.3f} R={a.recall:.3f} F1={a.fmeasure:.3f}")
print(f"Swapped -> P={b.precision:.3f} R={b.recall:.3f} F1={b.fmeasure:.3f}")

The **F-measure is symmetric**: when predictions and references are swapped, precision and recall are exchanged, but their harmonic mean (F1) does not change.

## Part VII. Comparing Small and Large Models (t5-small, t5-base, gpt2)

In [ ]:
def summarize_with_gpt2(articles, model_name="gpt2", batch_size=4,
                        max_new_tokens=60, max_context=1024):
    """Zero-shot summarization with GPT-2 using the 'TL;DR:' prompt."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"          # required for batched generation with decoder-only models
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()
    model.config.pad_token_id = tokenizer.eos_token_id

    suffix = "\nTL;DR:"
    suffix_len = len(tokenizer(suffix)["input_ids"])
    max_article_tokens = max_context - max_new_tokens - suffix_len   # GPT-2's 1024-token limit

    # Truncate each article so that prompt + generated tokens fit in the context window
    prompts = []
    for a in articles:
        ids = tokenizer(a, truncation=True, max_length=max_article_tokens)["input_ids"]
        prompts.append(tokenizer.decode(ids) + suffix)

    summaries = []
    for batch in batch_generator(prompts, batch_size):
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]   # keep only the generated part
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        summaries.extend([d.strip().split("\n")[0].strip() for d in decoded])

        del inputs, output_ids, new_tokens
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return summaries

In [ ]:
# Generate summaries with the other two models
t5_base_summaries = summarize_with_t5(articles, model_name="t5-base")
gpt2_summaries = summarize_with_gpt2(articles, model_name="gpt2")

model_summaries = {
    "t5-small": t5_small_summaries,
    "t5-base":  t5_base_summaries,
    "gpt2":     gpt2_summaries,
}
for name, s in model_summaries.items():
    print(f"{name}: {compute_rouge_score(s, references)}")

In [ ]:
def compute_rouge_per_row(predictions, references, use_stemmer=True):
    """DataFrame with the ROUGE scores of every individual article."""
    rows = [compute_rouge_score([p], [r], use_stemmer=use_stemmer)
            for p, r in zip(predictions, references)]
    return pd.DataFrame(rows)

per_row_scores = {name: compute_rouge_per_row(s, references)
                  for name, s in model_summaries.items()}

for name, df in per_row_scores.items():
    print(f"\n=== {name} (per-row ROUGE) ===")
    display(df.head(10))

## Part VIII. Comparing All Models

In [ ]:
def compare_models(per_row_scores):
    """Average the per-row ROUGE scores of every model into a single DataFrame."""
    return pd.DataFrame({name: df.mean() for name, df in per_row_scores.items()}).T.round(4)

def compare_models_summaries(articles, references, model_summaries, n=10):
    """Show the summaries of all models side by side."""
    data = {"reference": references}
    data.update(model_summaries)
    return pd.DataFrame(data).head(n)

comparison_df = compare_models(per_row_scores)
print("=== Average ROUGE per model ===")
display(comparison_df)

print("\n=== Side-by-side summaries ===")
display(compare_models_summaries(articles, references, model_summaries, n=10))

In [ ]:
# Comparison chart
comparison_df.plot(kind="bar", figsize=(9, 5))
plt.title("Average ROUGE per model")
plt.ylabel("F-measure")
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

## Analysis and Conclusions
- **Exact-match accuracy ≈ 0**: unsuitable for free-text generation; ROUGE measures partial overlap.
- **ROUGE-1 > ROUGE-2 > ROUGE-L** typically: matching single words is easier than matching consecutive bigrams.
- **t5-base** usually beats **t5-small** (more parameters → more faithful summaries); T5 was trained with the `summarize:` task.
- **gpt2** (zero-shot with `TL;DR:`) normally performs worse: it was not fine-tuned for summarization and its context is limited to 1024 tokens.
- ROUGE captures neither semantics nor fluency; it should be complemented with metrics such as BERTScore or human evaluation.

*Note:* complete your own analysis with the numbers you obtain in your run.